# Multivariable Calculus and Optimization Geometry
## Local linearity, Taylor models, Hessians, and constrained motion

### Learning goals

You will connect gradients to directional change, measure the improvement from second-order Taylor models, read Hessian eigenvalues as principal curvatures, diagnose gradient-descent stability, and solve one constrained problem geometrically.

Complete Notebook 19 first. Notebook 17 is recommended before the cumulative logistic-regression audit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.figsize': (7, 4.5), 'axes.grid': True})
rng = np.random.default_rng(7)

## 1. Gradients and directional change

For a differentiable scalar field, $D_vf(x)=\nabla f(x)^Tv$. Among unit directions, the gradient gives greatest increase and its negative gives greatest decrease. A contour is locally perpendicular to the gradient.

In [ ]:
def field(p):
    x, y = p
    return 0.5*(x*x + 8*y*y) + 0.4*x*y

def grad_field(p):
    x, y = p
    return np.array([x + 0.4*y, 8*y + 0.4*x])

p = np.array([1.5, -0.5])
v = np.array([3., 4.]); v /= np.linalg.norm(v)
h = 1e-5
directional = grad_field(p) @ v
finite = (field(p+h*v)-field(p-h*v))/(2*h)
assert np.isclose(directional, finite, rtol=1e-8)

xs = np.linspace(-3,3,80); ys=np.linspace(-2,2,80)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.5*(XX**2+8*YY**2)+0.4*XX*YY
plt.figure()
plt.contour(XX,YY,ZZ,levels=18)
g=grad_field(p)
plt.quiver(*p,*g,angles='xy',scale_units='xy',scale=1,color='C3',label='gradient')
plt.scatter(*p,color='black'); plt.axis('equal'); plt.title('Gradient crosses local contours'); plt.show()
print('analytic and finite-difference directional derivative:', directional, finite)

## 2. First- and second-order Taylor models

Near $x_0$, the first-order model uses value and gradient. The second-order model adds $\tfrac12\Delta^TH\Delta$. Shrinking-step experiments reveal approximation order more reliably than inspecting one point.

In [ ]:
def f(p):
    x,y=p; return np.sin(x)*np.exp(y)
def grad_f(p):
    x,y=p; return np.array([np.cos(x)*np.exp(y), np.sin(x)*np.exp(y)])
def hess_f(p):
    x,y=p; e=np.exp(y)
    return np.array([[-np.sin(x)*e, np.cos(x)*e], [np.cos(x)*e, np.sin(x)*e]])

x0=np.array([0.4,-0.3]); direction=np.array([1.,-2.]); direction/=np.linalg.norm(direction)
radii=np.logspace(-5,-0.3,70)
first_err=[]; second_err=[]
for radius in radii:
    d=radius*direction
    exact=f(x0+d)
    first=f(x0)+grad_f(x0)@d
    second=first+0.5*d@hess_f(x0)@d
    first_err.append(abs(exact-first)); second_err.append(abs(exact-second))
plt.figure()
plt.loglog(radii,first_err,label='first order error')
plt.loglog(radii,second_err,label='second order error')
plt.xlabel('step size'); plt.ylabel('absolute error'); plt.legend(); plt.show()
assert second_err[20] < first_err[20] / 1000
assert np.all(np.isfinite(second_err))

## 3. Hessian eigenvectors are principal curvature directions

For a quadratic $f(x)=\tfrac12x^THx$, the Hessian is constant. Positive eigenvalues mean upward curvature; a negative eigenvalue creates a descent direction at a stationary point. The eigenvalue ratio also controls first-order optimization difficulty.

In [ ]:
angle=np.deg2rad(30)
R=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])
H=R@np.diag([1.,25.])@R.T
eigenvalues,eigenvectors=np.linalg.eigh(H)
assert np.allclose(eigenvalues,[1,25])

grid=np.linspace(-2,2,120); XX,YY=np.meshgrid(grid,grid)
points=np.stack([XX,YY],axis=-1)
ZZ=0.5*np.einsum('...i,ij,...j->...',points,H,points)
plt.figure()
plt.contour(XX,YY,ZZ,levels=[.25,.5,1,2,4,8,16])
for value,vec,color in zip(eigenvalues,eigenvectors.T,['C2','C3']):
    plt.quiver(0,0,*vec,angles='xy',scale_units='xy',scale=1,color=color,label=f'curvature {value:g}')
plt.axis('equal'); plt.legend(); plt.title('Principal curvature directions'); plt.show()

## 4. Gradient-descent stability

In an eigendirection with curvature $\lambda$, one quadratic GD step multiplies the coordinate by $1-\eta\lambda$. Convergence requires $|1-\eta\lambda|<1$. Watch the steep direction oscillate before it diverges.

In [ ]:
def gd_path(start, eta, steps=35):
    path=[np.array(start,dtype=float)]
    for _ in range(steps): path.append(path[-1]-eta*(H@path[-1]))
    return np.array(path)

stable=gd_path([1.5,1.5],0.07)
unstable=gd_path([1.5,1.5],0.09,12)
plt.figure()
plt.contour(XX,YY,ZZ,levels=[.25,.5,1,2,4,8,16,32])
plt.plot(*stable.T,label='eta=.07 stable',marker='o',ms=2)
plt.plot(*unstable.T,label='eta=.09 unstable',marker='x',ms=3)
plt.xlim(-2,2);plt.ylim(-2,2);plt.axis('equal');plt.legend();plt.show()
assert np.linalg.norm(stable[-1]) < np.linalg.norm(stable[0]) / 5
assert np.linalg.norm(unstable[-1]) > np.linalg.norm(unstable[0])
assert np.isclose(2/eigenvalues[-1], 0.08)

## 5. Constrained optimization and projected gradients

To maximize $c^Tx$ subject to $\|x\|=1$, the optimum is $x=c/\|c\|$. At a constrained optimum, the objective gradient is parallel to the constraint normal; its tangent component is zero. Projected gradient ascent makes that geometry executable.

In [ ]:
c=np.array([1.,2.])
x=np.array([-1.,0.2]); x/=np.linalg.norm(x)
path=[x.copy()]
for _ in range(80):
    tangent_grad=c-x*(x@c)
    x=x+0.12*tangent_grad
    x/=np.linalg.norm(x)
    path.append(x.copy())
path=np.array(path); optimum=c/np.linalg.norm(c)
circle=np.vstack([np.cos(np.linspace(0,2*np.pi,300)),np.sin(np.linspace(0,2*np.pi,300))])
plt.figure()
plt.plot(*circle);plt.plot(*path.T,'o-',ms=2);plt.scatter(*optimum,s=80,label='analytic optimum')
plt.axis('equal');plt.legend();plt.title('Projected ascent on the unit circle');plt.show()
assert np.linalg.norm(path[-1]-optimum) < 1e-4
assert abs((c-optimum*(optimum@c))@np.array([-optimum[1],optimum[0]])) < 1e-12

## Cumulative logistic-regression geometry audit

For binary logistic regression, the Hessian is $X^TSX/n$, where $S$ contains $p_i(1-p_i)$. It is positive semidefinite, but can be singular or ill-conditioned when features are redundant. Verify the gradient two ways, inspect curvature, and compare a Taylor prediction with the true loss change.

In [ ]:
X_lr=rng.normal(size=(120,3)); X_lr[:,2]=X_lr[:,0]+0.02*rng.normal(size=120)
true_w=np.array([1.2,-0.8,0.3]); logits=X_lr@true_w
y_lr=rng.binomial(1,1/(1+np.exp(-logits)))

def logistic_parts(w):
    z=X_lr@w
    loss=np.mean(np.logaddexp(0,z)-y_lr*z)
    p=1/(1+np.exp(-z))
    grad=X_lr.T@(p-y_lr)/len(y_lr)
    hess=X_lr.T@(X_lr*(p*(1-p))[:,None])/len(y_lr)
    return loss,grad,hess

w=np.array([.2,-.1,.1]); loss,grad,hess=logistic_parts(w)
eps=1e-5
fd=np.array([(logistic_parts(w+eps*np.eye(3)[j])[0]-logistic_parts(w-eps*np.eye(3)[j])[0])/(2*eps) for j in range(3)])
step=-0.2*grad
actual=logistic_parts(w+step)[0]
taylor=loss+grad@step+0.5*step@hess@step
evals=np.linalg.eigvalsh(hess)
print('gradient check error:',np.linalg.norm(grad-fd))
print('Hessian eigenvalues:',evals,'condition:',evals[-1]/evals[0])
print('actual vs second-order predicted loss:',actual,taylor)
assert np.allclose(grad,fd,rtol=1e-6,atol=1e-8)
assert evals[0] >= -1e-12
assert evals[-1]/evals[0] > 1000
assert abs(actual-taylor) < 1e-4

### Cumulative explanation prompts

- Why is the gradient perpendicular to a contour?
- What empirical signature distinguishes first- from second-order approximation error?
- Why does the largest Hessian eigenvalue constrain the learning rate?
- How can a convex loss still have an ill-conditioned or singular Hessian?
- Which claim in this notebook depends on local behavior, and which is global?

Answer from memory and reproduce one finite-difference check without copying code.